# Задача 5: Трансформеры (RuBERT)
Обучение модели RuBERT-Tiny. Мы используем сырой текст (CONTENT) без лемматизации, так как BERT сам отлично понимает контекст.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report

# Загрузка данных
train_df = pd.read_parquet('../data/train.parquet')
test_df = pd.read_parquet('../data/test.parquet')

# БЕРЕМ СЫРОЙ ТЕКСТ (CONTENT)
train_texts = train_df['CONTENT'].astype(str).tolist()
test_texts = test_df['CONTENT'].astype(str).tolist()

# ВАЖНО: Нейросети требуют метки от 0 до N-1. 
# У нас классы от 1 до 8 (но 7 отсутствует). Сдвигаем их на -1.
train_labels = [int(label) - 1 for label in train_df['class']]
test_labels = [int(label) - 1 for label in test_df['class']]

# Количество уникальных классов (учитывая, что 8 -> 7)
num_labels = 8 

print(f"Данные для BERT готовы. Тренировка: {len(train_texts)}, Тест: {len(test_texts)}")

## Загрузка модели и токенизация
Используем легкую версию rubert-tiny2 для быстрого обучения.

In [ ]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels})

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

## Обучение и оценка

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, predictions, average="macro")}

training_args = TrainingArguments(
    output_dir="./results_rubert",
    num_train_epochs=10,            # <--- УВЕЛИЧИВАЕМ ДО 10 (пусть учится дольше)
    per_device_train_batch_size=8,  # <--- УМЕНЬШАЕМ БАТЧ (больше шагов обновления)
    per_device_eval_batch_size=16,
    learning_rate=5e-5,             # <--- Чуть увеличиваем скорость обучения
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=10,               # Чтобы видеть прогресс каждые 10 шагов
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

print("Начинаем обучение BERT...")
trainer.train()

## Финальная оценка

In [ ]:
results = trainer.evaluate()
print(f"\nMacro F1 (RuBERT-Tiny): {results['eval_macro_f1']:.4f}")

# Предсказания для отчета
predictions = trainer.predict(test_tokenized)
y_pred_bert = np.argmax(predictions.predictions, axis=-1)

print(classification_report(test_labels, y_pred_bert))

# Сохранение
results_df = pd.DataFrame({
    'Model': ['RuBERT-Tiny'],
    'Macro_F1': [results['eval_macro_f1']]
})
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/transformer_results.csv', index=False)
print("Результаты сохранены!")